In [ ]:
import os
from dotenv import load_dotenv
from datetime import datetime, timedelta, date
from zoneinfo import ZoneInfo

from google.cloud import storage

import pyspark.sql.functions as F
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DateType

from utils.PostParser import post_parser
from utils.LocationFunctions import get_locations_from_bq, get_missing_locations, get_batch_geocode, update_locations_bq

gcs_connector_path = '../../config/gcs-connector-hadoop3-latest.jar'
bigquery_connector_path = '../../config/spark-bigquery-with-dependencies_2.12-0.35.0.jar'

load_dotenv()

RawSchema = StructType([
    StructField('content', StringType(), True),
    StructField('tweetlinkid', StringType(), True),
    StructField('created_at', DateType(), True),
])

PartialPostSchema = StructType([
    StructField('Date', DateType(), True),
    StructField('Time', StringType(), True),
    StructField('Location', StringType(), True),
    StructField('Direction', StringType(), True),
    StructField('Type', StringType(), True),
    StructField('Lanes_Blocked', IntegerType(), True),
    StructField('Involved', StringType(), True),
    StructField('Tweet', StringType(), True),
    StructField('Source', StringType(), True),
])
  
# -------------------------------------------------------------------------------------------
def load_locations_df(spark, table_id):
    return spark.read.format("bigquery") \
                .option('table', table_id) \
                .load()

# -------------------------------------------------------------------------------------------
def generate_filenames(scrape_folder, start_date, end_date):
    """
    Generates list of GCS filenames between start_date and end_date (inclusive)
    """
    PHT = ZoneInfo("Asia/Manila")

    start = datetime.strptime(start_date, "%Y-%m-%d").replace(tzinfo=PHT)
    end = datetime.strptime(end_date, "%Y-%m-%d").replace(tzinfo=PHT)

    current = start
    filenames = []

    while current <= end:
        year = current.strftime("%Y")
        month = current.strftime("%m")
        day = current.strftime("%d")

        filename = f"{scrape_folder}/{year}/{month}/scrape_data_{year}{month}{day}.csv"
        filenames.append(filename)

        current += timedelta(days=1)

    return filenames
   

# -------------------------------------------------------------------------------------------
def gcs_file_read(spark, bucket_name, filename):
    df = spark.read \
        .option("header", True) \
        .option("multiline", True) \
        .option("quote", '"') \
        .option("escape", '"') \
        .option("ignoreLeadingWhiteSpace", True) \
        .option("ignoreTrailingWhiteSpace", True) \
        .schema(RawSchema) \
        .csv(f"gs://{bucket_name}/{filename}")
    
    return df

# -------------------------------------------------------------------------------------------
def gcs_upload_parquet(bucket_name, clean_folder, df):
    df.write \
        .mode("append") \
        .partitionBy("Date") \
        .parquet(f"gs://{bucket_name}/{clean_folder}/")

# -------------------------------------------------------------------------------------------
def partial_parse_raw_data(df_raw):
    post_parser_udf = F.udf(post_parser, PartialPostSchema)

    df_temp = df_raw.withColumn("parsed", 
                                post_parser_udf(
                                    F.upper(df_raw['content']),
                                    df_raw['created_at'],
                                    df_raw['tweetlinkid']
                                    )
                                )
    
    return df_temp.select(
        F.col("parsed.Date").alias("Date"),
        F.col("parsed.Time").alias("Time"),
        F.col("parsed.Location").alias("Location"),
        F.col("parsed.Direction").alias("Direction"),
        F.col("parsed.Type").alias("Type"),
        F.col("parsed.Lanes_Blocked").alias("Lanes_Blocked"),
        F.col("parsed.Involved").alias("Involved"),
        F.col("parsed.Tweet").alias("Tweet"),
        F.col("parsed.Source").alias("Source")
    )

In [3]:
spark = SparkSession.builder \
    .master("local[*]") \
    .appName('Transform Stage') \
    .config("spark.jars", f"{gcs_connector_path},{bigquery_connector_path}") \
    .config("spark.hadoop.fs.gs.impl", "com.google.cloud.hadoop.fs.gcs.GoogleHadoopFileSystem") \
    .config("spark.hadoop.google.cloud.auth.service.account.enable", "true") \
    .getOrCreate() 

gcs_client = storage.Client()

project_id = os.getenv("PROJECT_ID")
dataset = os.getenv("DATASET")
table_id = f"{project_id}:{dataset}.locations"
bucket_name = os.getenv('BUCKET_NAME')
raw_folder = os.getenv('RAW_FOLDER_NAME')
clean_folder = os.getenv('CLEANED_FOLDER_NAME')

scrape_folder = f"{raw_folder}/scrape"

bucket = gcs_client.bucket(bucket_name)

# Load locations once (reuse for all files)
df_locations = load_locations_df(spark, table_id)

26/03/29 22:15:33 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


In [4]:
start_date = "2026-03-15"
end_date = "2026-03-20"

In [5]:
filenames = generate_filenames(scrape_folder, start_date, end_date)
filenames

['raw/scrape/2026/03/scrape_data_20260315.csv',
 'raw/scrape/2026/03/scrape_data_20260316.csv',
 'raw/scrape/2026/03/scrape_data_20260317.csv',
 'raw/scrape/2026/03/scrape_data_20260318.csv',
 'raw/scrape/2026/03/scrape_data_20260319.csv',
 'raw/scrape/2026/03/scrape_data_20260320.csv']

In [6]:
raw_filename = filenames[0]
print(f"Processing: {raw_filename}")

blob = storage.Blob(bucket=bucket, name=raw_filename)

if not blob.exists():
    print(f"Skipping (not found): {raw_filename}")

df_raw = gcs_file_read(spark, bucket_name, raw_filename)

Processing: raw/scrape/2026/03/scrape_data_20260315.csv


In [7]:
df_partial_parsed = partial_parse_raw_data(df_raw)

In [8]:
df_partial_parsed.show(5, truncate=False)

+----------+-------+--------------------------------------+---------+-------------------+-------------+------------------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-----------------------------------------------+
|Date      |Time   |Location                              |Direction|Type               |Lanes_Blocked|Involved          |Tweet                                                                                                                                                                                |Source                                         |
+----------+-------+--------------------------------------+---------+-------------------+-------------+------------------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [9]:
df_full_parsed = get_locations_from_bq(df_locations, df_partial_parsed)

In [ ]:
df_full_parsed.show(5, truncate=False)

In [ ]:
df_full_parsed.filter(F.col("Location") == '').show(truncate=False)

In [10]:
missing_locations = get_missing_locations(df_full_parsed)

In [11]:
missing_locations

['COMMONWEALTH CAMARO',
 'EAST BANK ROAD FLOODWAY BRGY MANGGAHAN',
 'QUEZON AVE. FISHER MALL',
 'B. SERRANO KATIPUNAN',
 'COMMONWEALTH ECLARO U-TURN SLOT',
 'MERALCO AVENUE INFRONT OF MRA',
 'SALES ROAD ROTONDA',
 'PRESIDENT QUIRINO MABINI BRIDGE',
 'PAYATAS ROAD ROLLING HILLS',
 'COMMONWEALTH ST. PETER',
 'QUIRINO AVENUE SAN ANDRES',
 'DOMESTIC ROAD BEFORE ANDREWS',
 'MARCOS HIGHWAY METRO EAST',
 'J. VARGAS MERALCO',
 'DEL MONTE ARANETA',
 'KATIPUNAN MIRRIAM']

In [ ]:
len(missing_locations)

In [18]:
resolved_locations_df = get_batch_geocode(spark, missing_locations)

Retry failed for MERALCO AVENUE INFRONT OF MRA: 'locality'
Retry failed for MERALCO AVENUE INFRONT OF MRA: 'locality'


In [13]:
resolved_locations_df.show(5, truncate=False)

+-----------+--------------------------------------+----------------+----------------+-------------+
|City       |Location                              |Latitude        |Longitude       |High_Accuracy|
+-----------+--------------------------------------+----------------+----------------+-------------+
|Quezon City|COMMONWEALTH CAMARO                   |14.7033353500222|121.070975349869|0.0          |
|Manila     |EAST BANK ROAD FLOODWAY BRGY MANGGAHAN|14.5754623000362|121.102220500124|0.5          |
|Quezon City|QUEZON AVE. FISHER MALL               |14.6370686500856|121.025786199883|0.0          |
|Taguig     |B. SERRANO KATIPUNAN                  |14.5583896873721|121.064401565757|0.5          |
|Taytay     |COMMONWEALTH ECLARO U-TURN SLOT       |14.5670364000052|121.141459750006|0.5          |
+-----------+--------------------------------------+----------------+----------------+-------------+
only showing top 5 rows



In [14]:
resolved_locations_df.filter(F.col("City") == '').show(truncate=False)

+----+-----------------------------+-----------------+------------------+-------------+
|City|Location                     |Latitude         |Longitude         |High_Accuracy|
+----+-----------------------------+-----------------+------------------+-------------+
|    |MERALCO AVENUE INFRONT OF MRA|14.57361125946045|121.03297424316406|0.5          |
+----+-----------------------------+-----------------+------------------+-------------+



In [ ]:
missing_cities_df = resolved_locations_df.filter(F.col("City") == '')
missing_cities_df.show(truncate=False)

In [ ]:
missing_cities_df = missing_cities_df.select("Location", "Latitude", "Longitude", "High_Accuracy").distinct()

In [ ]:
missing_cities_df.show(truncate=False)

In [15]:
update_locations_bq(spark, df_locations, resolved_locations_df, table_id)

In [16]:
df_full_parsed = get_locations_from_bq(df_locations, df_full_parsed)

In [17]:
df_full_parsed.show(5, truncate=False)

+--------------------------------------+-----------+----------------+----------------+-------------+-----------+----------------+----------------+-------------+----------+-------+---------+-------------------+-------------+------------------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-----------------------------------------------+
|Location                              |City       |Latitude        |Longitude       |High_Accuracy|City       |Latitude        |Longitude       |High_Accuracy|Date      |Time   |Direction|Type               |Lanes_Blocked|Involved          |Tweet                                                                                                                                                                                |Source                                         |
+--------------------------------------+-----------+--